In [35]:
import os 
import sys
import requests
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns   
import math
import scipy.stats as stats
from io import StringIO
from datetime import datetime, timedelta
import warnings 
warnings.filterwarnings("ignore")

In [36]:
# 폰트 깨짐 방지
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic') 
else:
    
    plt.rc('font', family='NanumGothic')

# 마이너스 기호방지
plt.rcParams['axes.unicode_minus'] = False

In [37]:
articles = pd.read_csv("articles.csv", encoding= 'utf-8')
customers = pd.read_csv("customers.csv", encoding= 'utf-8')
submission = pd.read_csv("sample_submission.csv", encoding= 'utf-8')
transactions = pd.read_csv("transactions_train.csv", encoding= 'utf-8')

## 데이터 결합

In [38]:
# 1. 거래 데이터 + 상품 데이터 조인
df = pd.merge(transactions, articles, on='article_id', how='left')

# 2. 거래/상품 데이터 + 고객 데이터 조인
df = pd.merge(df, customers, on='customer_id', how='left')

# 3. 최종 결과에 Submission 테이블 조인
df = pd.merge(df, submission, on='customer_id', how='left')

print(df.head())

        t_dat                                        customer_id  article_id  \
0  2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   663713001   
1  2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   541518023   
2  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   505221004   
3  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   685687003   
4  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   685687004   

      price  sales_channel_id  product_code                 prod_name  \
0  0.050831                 2        663713  Atlanta Push Body Harlow   
1  0.030492                 2        541518   Rae Push (Melbourne) 2p   
2  0.015237                 2        505221               Inca Jumper   
3  0.016932                 2        685687      W YODA KNIT OL OFFER   
4  0.016932                 2        685687      W YODA KNIT OL OFFER   

   product_type_no product_type_name  product_group_name  ...  \
0              

## 값 수정

In [39]:
df['price'] = df['price'] * 590

df.loc[(df['FN'].isna()) & (df['fashion_news_frequency'] == 'Regularly'), 'FN'] = 1
df.loc[(df['FN'].isna()) & (df['fashion_news_frequency'] == 'Monthly'), 'FN'] = 1

## 칼럼 삭제

In [40]:
cols_to_drop = [
    'detail_desc', 'postal_code', 'product_code',
    'product_type_no', 'graphical_appearance_no', 'colour_group_code', 
    'perceived_colour_value_id', 'perceived_colour_master_id', 
    'department_no', 'index_code', 'index_group_no', 'section_no', 'garment_group_no'
]

df.drop(columns=cols_to_drop, inplace=True)

print(f"남은 칼럼 수: {len(df.columns)}")

남은 칼럼 수: 23


## 결측치 처리

In [41]:
df['FN'] = df['FN'].fillna(0)

df['Active'] = df['Active'].fillna(0)

df['club_member_status'] = df['club_member_status'].fillna('Unknown')

df['fashion_news_frequency'] = df['fashion_news_frequency'].replace('None', 'NONE')
df['fashion_news_frequency'] = df['fashion_news_frequency'].fillna('NONE')

df = df.dropna(subset=['age'])

## 타입 변환

In [42]:
# 1. Age (나이): 실수에서 정수로 변환
df['age'] = df['age'].astype(int)

# 2. 범주형 데이터
# 'club_member_status'나 'fashion_news_frequency'처럼 반복되는 글자는 'category' 타입으로 바꾸면 메모리 사용량이 80% 이상 줄어든다고 함
df['club_member_status'] = df['club_member_status'].astype('category')
df['fashion_news_frequency'] = df['fashion_news_frequency'].astype('category')

# 3. ID 칼럼: 문자열로 통일 ('0'이 잘리는 문제를 방지하기 위해 문자열로 고정)
df['article_id'] = df['article_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)

# 4. 날짜 데이터: 문자열에서 날짜형으로 변환
df['t_dat'] = pd.to_datetime(df['t_dat'])


df['FN'] = df['FN'].astype(int)
df['Active'] = df['Active'].astype(int)

## 칼럼 생성

In [43]:
# 1. 구간 설정 (0~19세, 20~29세, ..., 59세~150세)
bins = [0, 20, 30, 40, 50, 60, 150]

# 2. 라벨 설정 (각 구간의 이름)
labels = ['10대', '20대', '30대', '40대', '50대', '60대 이상']

# 3. 새로운 칼럼 생성
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# 확인
print(df[['age', 'age_group']].head())

   age age_group
0   24       20대
1   24       20대
2   32       30대
3   32       30대
4   32       30대


## 확인

In [44]:
df.info()

<class 'pandas.DataFrame'>
Index: 31648066 entries, 0 to 31788323
Data columns (total 24 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   t_dat                         datetime64[us]
 1   customer_id                   str           
 2   article_id                    str           
 3   price                         float64       
 4   sales_channel_id              int64         
 5   prod_name                     str           
 6   product_type_name             str           
 7   product_group_name            str           
 8   graphical_appearance_name     str           
 9   colour_group_name             str           
 10  perceived_colour_value_name   str           
 11  perceived_colour_master_name  str           
 12  department_name               str           
 13  index_name                    str           
 14  index_group_name              str           
 15  section_name                  str           
 

In [45]:
missing_counts = df.isnull().sum()

print(missing_counts[missing_counts == 0])

t_dat                           0
customer_id                     0
article_id                      0
price                           0
sales_channel_id                0
prod_name                       0
product_type_name               0
product_group_name              0
graphical_appearance_name       0
colour_group_name               0
perceived_colour_value_name     0
perceived_colour_master_name    0
department_name                 0
index_name                      0
index_group_name                0
section_name                    0
garment_group_name              0
FN                              0
Active                          0
club_member_status              0
fashion_news_frequency          0
age                             0
prediction                      0
age_group                       0
dtype: int64


## 저장

In [ ]:
# drop=True: 기존의 뒤섞인 인덱스를 삭제
# inplace=True: 변경 사항을 현재 데이터프레임에 바로 적용
df.reset_index(drop=True, inplace=True)

In [ ]:
# index=False: 저장할 때 인덱스 번호를 파일에 포함하지 않음
df.to_csv('clean_data.csv', index=False)